In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load cleaned dataset

file_path = r"C:\Users\Sandeep\OneDrive\Documents\Projects\IPL-Performance-Intelligence\data\cleaned\ipl_cleaned.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (295732, 57)


In [3]:
print("Columns available:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Columns available:

1. match_id
2. date
3. match_type
4. event_name
5. innings
6. batting_team
7. bowling_team
8. over
9. ball
10. ball_no
11. batter
12. bat_pos
13. runs_batter
14. balls_faced
15. bowler
16. valid_ball
17. runs_extras
18. runs_total
19. runs_bowler
20. runs_not_boundary
21. extra_type
22. non_striker
23. non_striker_pos
24. wicket_kind
25. player_out
26. fielders
27. runs_target
28. umpires_call
29. player_of_match
30. match_won_by
31. win_outcome
32. toss_winner
33. toss_decision
34. venue
35. city
36. day
37. month
38. year
39. season
40. gender
41. team_type
42. superover_winner
43. result_type
44. method
45. balls_per_over
46. overs
47. event_match_no
48. stage
49. match_number
50. team_runs
51. team_balls
52. team_wicket
53. batter_runs
54. batter_balls
55. bowler_wicket
56. batting_partners
57. striker_out


In [4]:
# Feature Engineering: Innings Phase

def get_innings_phase(over):
    if over < 6:
        return "Powerplay"
    elif over < 15:
        return "Middle"
    else:
        return "Death"

df["innings_phase"] = df["over"].apply(get_innings_phase)

print("Innings phase feature created successfully!")
print(df["innings_phase"].value_counts())

Innings phase feature created successfully!
innings_phase
Middle       135655
Powerplay     92904
Death         67173
Name: count, dtype: int64


In [5]:
# Ball outcome features

# Dot ball
df["is_dot_ball"] = (
    (df["runs_total"] == 0) &
    (df["valid_ball"] == 1)
).astype(int)

# Boundary indicators
df["is_four"] = (df["runs_batter"] == 4).astype(int)

df["is_six"] = (df["runs_batter"] == 6).astype(int)

# Any boundary
df["is_boundary"] = (
    (df["is_four"] == 1) |
    (df["is_six"] == 1)
).astype(int)

# Wicket indicator
df["is_wicket"] = df["wicket_kind"].notna().astype(int)

print("Ball outcome features created successfully!")

df[
    [
        "is_dot_ball",
        "is_four",
        "is_six",
        "is_boundary",
        "is_wicket"
    ]
].sum()

Ball outcome features created successfully!


is_dot_ball    101230
is_four         34447
is_six          15779
is_boundary     50226
is_wicket       14705
dtype: int64

In [6]:
# Feature Engineering: Extras Indicators

df["is_wide"] = df["extra_type"].astype(str).str.contains(
    "wides",
    na=False
).astype(int)

df["is_no_ball"] = df["extra_type"].astype(str).str.contains(
    "noballs",
    na=False
).astype(int)

df["is_bye"] = df["extra_type"].astype(str).str.contains(
    "byes",
    na=False
).astype(int)

df["is_leg_bye"] = df["extra_type"].astype(str).str.contains(
    "legbyes",
    na=False
).astype(int)

print("Extras features created successfully!")

df[
    ["is_wide", "is_no_ball", "is_bye", "is_leg_bye"]
].sum()

Extras features created successfully!


is_wide       9876
is_no_ball    1226
is_bye        5151
is_leg_bye    4406
dtype: int64

In [7]:
# Feature Engineering: Batter Run Categories

df["is_single"] = (df["runs_batter"] == 1).astype(int)

df["is_double"] = (df["runs_batter"] == 2).astype(int)

df["is_triple"] = (df["runs_batter"] == 3).astype(int)

df["is_zero_batter_run"] = (
    df["runs_batter"] == 0
).astype(int)

print("Batter run category features created successfully!")

df[
    [
        "is_zero_batter_run",
        "is_single",
        "is_double",
        "is_triple",
        "is_four",
        "is_six"
    ]
].sum()

Batter run category features created successfully!


is_zero_batter_run    116759
is_single             109492
is_double              18338
is_triple                843
is_four                34447
is_six                 15779
dtype: int64

In [8]:
# Check all newly created feature columns

feature_cols = [
    'innings_phase',
    'is_dot_ball',
    'is_four',
    'is_six',
    'is_boundary',
    'is_wicket',
    'is_wide',
    'is_no_ball',
    'is_bye',
    'is_leg_bye',
    'is_zero_batter_run',
    'is_single',
    'is_double',
    'is_triple'
]

df[feature_cols].head()

,innings_phase,is_dot_ball,is_four,is_six,is_boundary,is_wicket,is_wide,is_no_ball,is_bye,is_leg_bye,is_zero_batter_run,is_single,is_double,is_triple
0,Powerplay,0,0,0,0,0,0,0,1,1,1,0,0,0
1,Powerplay,1,0,0,0,0,0,0,0,0,1,0,0,0
2,Powerplay,0,0,0,0,0,1,0,0,0,1,0,0,0
3,Powerplay,1,0,0,0,0,0,0,0,0,1,0,0,0
4,Powerplay,1,0,0,0,0,0,0,0,0,1,0,0,0


In [9]:
# Validate mutually exclusive extras

print("Bye + Leg Bye conflicts:",
      ((df['is_bye'] == 1) & (df['is_leg_bye'] == 1)).sum())

print("Wide + No Ball conflicts:",
      ((df['is_wide'] == 1) & (df['is_no_ball'] == 1)).sum())

# Check feature combinations
print("\nDot balls:", df['is_dot_ball'].sum())
print("Boundaries:", df['is_boundary'].sum())
print("Wickets:", df['is_wicket'].sum())

Bye + Leg Bye conflicts: 4406
Wide + No Ball conflicts: 0

Dot balls: 101230
Boundaries: 50226
Wickets: 14705


In [10]:
# Fix extras features

df['is_wide'] = df['extra_type'].astype(str).str.contains(
    r'\bwides\b',
    case=False,
    na=False
).astype(int)

df['is_no_ball'] = df['extra_type'].astype(str).str.contains(
    r'\bnoballs\b',
    case=False,
    na=False
).astype(int)

df['is_leg_bye'] = df['extra_type'].astype(str).str.contains(
    r'\blegbyes\b',
    case=False,
    na=False
).astype(int)

# Exact check for byes without accidentally matching legbyes
df['is_bye'] = (
    df['extra_type']
    .fillna('')
    .str.lower()
    .str.split(',')
    .apply(lambda x: 'byes' in [item.strip() for item in x])
    .astype(int)
)

print("Extras features corrected successfully!")

print("\nFeature counts:")
print(df[['is_wide', 'is_no_ball', 'is_bye', 'is_leg_bye']].sum())

Extras features corrected successfully!

Feature counts:
is_wide       9876
is_no_ball    1226
is_bye         745
is_leg_bye    4406
dtype: int64


In [11]:
print("Bye + Leg Bye conflicts:",
      ((df['is_bye'] == 1) & (df['is_leg_bye'] == 1)).sum())

print("Wide + No Ball conflicts:",
      ((df['is_wide'] == 1) & (df['is_no_ball'] == 1)).sum())

Bye + Leg Bye conflicts: 0
Wide + No Ball conflicts: 0


In [12]:
# Feature Engineering: Match Context

# A team is considered to be chasing when a runs target is available
df['is_chasing'] = df['runs_target'].notna().astype(int)

print("Chasing feature created successfully!")

print(df['is_chasing'].value_counts())

Chasing feature created successfully!
is_chasing
0    153429
1    142303
Name: count, dtype: int64


In [13]:
# Inspect chase-related columns

chasing_sample = df.loc[
    df['is_chasing'] == 1,
    [
        'match_id',
        'innings',
        'over',
        'ball',
        'batting_team',
        'runs_total',
        'team_runs',
        'team_balls',
        'runs_target'
    ]
].head(10)

chasing_sample

,match_id,innings,over,ball,batting_team,runs_total,team_runs,team_balls,runs_target
124,335982,2,0,1,Royal Challengers Bengaluru,1,1,1,223.0
125,335982,2,0,2,Royal Challengers Bengaluru,1,2,1,223.0
126,335982,2,0,2,Royal Challengers Bengaluru,0,2,2,223.0
127,335982,2,0,3,Royal Challengers Bengaluru,1,3,3,223.0
128,335982,2,0,4,Royal Challengers Bengaluru,1,4,4,223.0
129,335982,2,0,5,Royal Challengers Bengaluru,0,4,5,223.0
130,335982,2,0,6,Royal Challengers Bengaluru,0,4,6,223.0
131,335982,2,1,1,Royal Challengers Bengaluru,0,4,7,223.0
132,335982,2,1,2,Royal Challengers Bengaluru,0,4,8,223.0
133,335982,2,1,3,Royal Challengers Bengaluru,4,8,9,223.0


In [14]:
# Feature Engineering: Chase Pressure

# Runs still needed to reach the target
df['runs_remaining'] = df['runs_target'] - df['team_runs']

# Total balls available in the innings
df['total_balls_available'] = df['overs'] * df['balls_per_over']

# Valid balls still remaining
df['balls_remaining'] = (
    df['total_balls_available'] - df['team_balls']
)

print("Chase pressure features created successfully!")

test = df.loc[
    df['is_chasing']==1,
    [
        'runs_target',
        'team_runs',
        'runs_remaining',
        'overs',
        'balls_per_over',
        'team_balls',
        'balls_remaining'
    ]

].head(10)
test

Chase pressure features created successfully!


,runs_target,team_runs,runs_remaining,overs,balls_per_over,team_balls,balls_remaining
124,223.0,1,222.0,20,6,1,119
125,223.0,2,221.0,20,6,1,119
126,223.0,2,221.0,20,6,2,118
127,223.0,3,220.0,20,6,3,117
128,223.0,4,219.0,20,6,4,116
129,223.0,4,219.0,20,6,5,115
130,223.0,4,219.0,20,6,6,114
131,223.0,4,219.0,20,6,7,113
132,223.0,4,219.0,20,6,8,112
133,223.0,8,215.0,20,6,9,111


In [15]:
# Feature Engineering: Required Run Rate

# Required runs per over to reach the target
df['required_run_rate'] = (
    df['runs_remaining'] / df['balls_remaining']
) * df['balls_per_over']

print("Required Run Rate feature created successfully!")

# Validate on chasing rows
test = df.loc[
    df['is_chasing'] == 1,
    [
        'runs_remaining',
        'balls_remaining',
        'balls_per_over',
        'required_run_rate'
    ]
].head(10)

test

Required Run Rate feature created successfully!


,runs_remaining,balls_remaining,balls_per_over,required_run_rate
124,222.0,119,6,11.193277
125,221.0,119,6,11.142857
126,221.0,118,6,11.237288
127,220.0,117,6,11.282051
128,219.0,116,6,11.327586
129,219.0,115,6,11.426087
130,219.0,114,6,11.526316
131,219.0,113,6,11.628319
132,219.0,112,6,11.732143
133,215.0,111,6,11.621622


In [16]:
# Check for rows where no balls are remaining
zero_balls_remaining = df[
    (df['is_chasing'] == 1) &
    (df['balls_remaining'] == 0)
]

print("Rows with 0 balls remaining:", len(zero_balls_remaining))

zero_balls_remaining[
    [
        'match_id',
        'team_runs',
        'runs_target',
        'runs_remaining',
        'balls_remaining',
        'required_run_rate'
    ]
].head(10)

Rows with 0 balls remaining: 463


,match_id,team_runs,runs_target,runs_remaining,balls_remaining,required_run_rate
472,335983,207,241.0,34.0,0,inf
1878,335989,202,209.0,7.0,0,inf
2376,335991,116,183.0,67.0,0,inf
4021,335998,181,192.0,11.0,0,inf
5000,336003,169,179.0,10.0,0,inf
5244,336034,153,157.0,4.0,0,inf
6618,336009,188,188.0,0.0,0,NaN
7291,336013,163,182.0,19.0,0,inf
7543,336014,181,205.0,24.0,0,inf
8715,336020,182,195.0,13.0,0,inf


In [17]:
import numpy as np

# Replace infinite required run rates with NaN
df['required_run_rate'] = df['required_run_rate'].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite Required Run Rate values:", 
      np.isinf(df['required_run_rate']).sum())

print("Missing Required Run Rate values:",
      df['required_run_rate'].isna().sum())

Infinite Required Run Rate values: 0
Missing Required Run Rate values: 153892


In [18]:
# Check rows where no valid balls have been faced yet
zero_team_balls = df[df['team_balls'] == 0]

print("Rows with team_balls = 0:", len(zero_team_balls))

zero_team_balls[
    [
        'match_id',
        'innings',
        'over',
        'ball',
        'team_runs',
        'team_balls'
    ]
].head(10)

Rows with team_balls = 0: 165


,match_id,innings,over,ball,team_runs,team_balls
1624,335989,1,0,1,1,0
5123,336034,2,0,1,1,0
5692,336006,1,0,1,1,0
7913,336016,2,0,1,2,0
9603,336025,2,0,1,1,0
9604,336025,2,0,1,2,0
9890,336027,1,0,1,2,0
10238,336028,2,0,1,1,0
10239,336028,2,0,1,6,0
10240,336028,2,0,1,7,0


In [19]:
# Feature Engineering: Current Run Rate

# Replace 0 balls with NaN to avoid division by zero
safe_team_balls = df['team_balls'].replace(0, np.nan)

# Calculate current scoring rate per over
df['current_run_rate'] = (
    df['team_runs'] / safe_team_balls
) * df['balls_per_over']

print("Current Run Rate feature created successfully!")

# Check for infinite values
print(
    "Infinite Current Run Rate values:",
    np.isinf(df['current_run_rate']).sum()
)

# Preview the feature
test = df[
    [
        'team_runs',
        'team_balls',
        'balls_per_over',
        'current_run_rate'
    ]
].head(10)

test

Current Run Rate feature created successfully!
Infinite Current Run Rate values: 0


,team_runs,team_balls,balls_per_over,current_run_rate
0,1,1,6,6.000000
1,1,2,6,3.000000
2,2,2,6,6.000000
3,2,3,6,4.000000
4,2,4,6,3.000000
5,2,5,6,2.400000
6,3,6,6,3.000000
7,3,7,6,2.571429
8,7,8,6,5.250000
9,11,9,6,7.333333


In [20]:
# Feature Engineering: Run Rate Pressure

# Difference between the required scoring rate and current scoring rate
df['run_rate_pressure'] = (
    df['required_run_rate'] - df['current_run_rate']
)

print("Run Rate Pressure feature created successfully!")

# Preview chasing rows
test = df.loc[
    df['is_chasing'] == 1,
    [
        'team_runs',
        'runs_remaining',
        'current_run_rate',
        'required_run_rate',
        'run_rate_pressure'
    ]
].head(10)

test

Run Rate Pressure feature created successfully!


,team_runs,runs_remaining,current_run_rate,required_run_rate,run_rate_pressure
124,1,222.0,6.000000,11.193277,5.193277
125,2,221.0,12.000000,11.142857,-0.857143
126,2,221.0,6.000000,11.237288,5.237288
127,3,220.0,6.000000,11.282051,5.282051
128,4,219.0,6.000000,11.327586,5.327586
129,4,219.0,4.800000,11.426087,6.626087
130,4,219.0,4.000000,11.526316,7.526316
131,4,219.0,3.428571,11.628319,8.199747
132,4,219.0,3.000000,11.732143,8.732143
133,8,215.0,5.333333,11.621622,6.288288


In [21]:
# Check wicket limits in the dataset

print("Maximum wickets lost:", df['team_wicket'].max())

print("\nWicket values:")
print(df['team_wicket'].value_counts().sort_index())

Maximum wickets lost: 10

Wicket values:
team_wicket
0     58972
1     57922
2     52145
3     43484
4     33297
5     21453
6     13427
7      7620
8      4659
9      2526
10      227
Name: count, dtype: int64


In [22]:
# Feature Engineering: Wickets Remaining

# A team can lose a maximum of 10 wickets
df['wickets_remaining'] = 10 - df['team_wicket']

print("Wickets remaining feature created successfully!")

# Check the distribution
print(df['wickets_remaining'].value_counts().sort_index())

Wickets remaining feature created successfully!
wickets_remaining
0       227
1      2526
2      4659
3      7620
4     13427
5     21453
6     33297
7     43484
8     52145
9     57922
10    58972
Name: count, dtype: int64


In [23]:
# Feature Engineering: Wicket Pressure

# Calculate the proportion of wickets already lost
df['wicket_pressure'] = df['team_wicket'] / 10

print("Wicket pressure feature created successfully!")

# Check the first few values
df[
    [
        'team_wicket',
        'wickets_remaining',
        'wicket_pressure'
    ]
].head(10)

Wicket pressure feature created successfully!


,team_wicket,wickets_remaining,wicket_pressure
0,0,10,0.0
1,0,10,0.0
2,0,10,0.0
3,0,10,0.0
4,0,10,0.0
5,0,10,0.0
6,0,10,0.0
7,0,10,0.0
8,0,10,0.0
9,0,10,0.0


In [24]:
df[
    [
        'team_wicket',
        'wickets_remaining',
        'wicket_pressure'
    ]
].drop_duplicates().sort_values('team_wicket')

,team_wicket,wickets_remaining,wicket_pressure
0,0,10,0.0
33,1,9,0.1
74,2,8,0.2
106,3,7,0.3
157,4,6,0.4
174,5,5,0.5
177,6,4,0.6
183,7,3,0.7
197,8,2,0.8
210,9,1,0.9


In [25]:
# Inspect pressure feature distributions

pressure_columns = [
    'required_run_rate',
    'wicket_pressure',
    'run_rate_pressure'
]

df[pressure_columns].describe()

,required_run_rate,wicket_pressure,run_rate_pressure
count,141840.000000,295732.000000,141755.000000
mean,10.826505,0.246607,2.961993
std,13.312951,0.210523,13.612698
min,-30.000000,0.000000,-50.369748
25%,7.304348,0.100000,-1.160884
50%,9.096774,0.200000,1.520000
75%,11.312500,0.400000,4.526786
max,792.000000,1.000000,785.344538


In [26]:
# Inspect extreme Required Run Rate values

extreme_rrr = df[
    (df['required_run_rate'] < 0) |
    (df['required_run_rate'] > 30)
]

print("Number of extreme Required Run Rate rows:", len(extreme_rrr))

extreme_rrr[
    [
        'match_id',
        'innings',
        'over',
        'ball',
        'team_runs',
        'runs_target',
        'runs_remaining',
        'team_balls',
        'balls_remaining',
        'required_run_rate'
    ]
].head(20)

Number of extreme Required Run Rate rows: 3543


,match_id,innings,over,ball,team_runs,runs_target,runs_remaining,team_balls,balls_remaining,required_run_rate
463,335983,2,18,3,194,241.0,47.0,111,9,31.333333
464,335983,2,18,4,195,241.0,46.0,112,8,34.500000
465,335983,2,18,5,197,241.0,44.0,113,7,37.714286
466,335983,2,18,6,198,241.0,43.0,114,6,43.000000
467,335983,2,19,1,199,241.0,42.0,115,5,50.400000
468,335983,2,19,2,200,241.0,41.0,116,4,61.500000
469,335983,2,19,3,200,241.0,41.0,117,3,82.000000
470,335983,2,19,4,206,241.0,35.0,118,2,105.000000
471,335983,2,19,5,207,241.0,34.0,119,1,204.000000
691,335984,2,15,1,132,130.0,-2.0,91,29,-0.413793


In [27]:
# Correct Runs Remaining

# A team cannot need fewer than 0 runs
df['runs_remaining'] = (
    df['runs_target'] - df['team_runs']
).clip(lower=0)

print("Runs remaining corrected successfully!")

# Validate the minimum value
print("Minimum runs remaining:", df['runs_remaining'].min())

Runs remaining corrected successfully!
Minimum runs remaining: 0.0


In [28]:
# Recalculate Required Run Rate

df['required_run_rate'] = (
    df['runs_remaining'] / df['balls_remaining']
) * df['balls_per_over']

# Replace infinite values with NaN
df['required_run_rate'] = df['required_run_rate'].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Required Run Rate recalculated successfully!")

# Validate the updated values
print(df['required_run_rate'].describe())

Required Run Rate recalculated successfully!
count    141840.000000
mean         10.835747
std          13.301867
min           0.000000
25%           7.304348
50%           9.096774
75%          11.312500
max         792.000000
Name: required_run_rate, dtype: float64


In [29]:
# Validate chase pressure features

chase_features = [
    'is_chasing',
    'runs_target',
    'team_runs',
    'runs_remaining',
    'team_balls',
    'balls_remaining',
    'current_run_rate',
    'required_run_rate',
    'run_rate_pressure'
]

df[chase_features].describe()

,is_chasing,runs_target,team_runs,runs_remaining,team_balls,balls_remaining,current_run_rate,required_run_rate,run_rate_pressure
count,295732.000000,142303.000000,295732.000000,142303.000000,295732.000000,295732.000000,295567.000000,141840.000000,141755.000000
mean,0.481189,171.548400,78.146291,95.298553,58.592844,61.407156,7.786352,10.835747,2.961993
std,0.499647,33.079009,50.698248,52.058153,34.106127,34.106127,2.509778,13.301867,13.612698
min,0.000000,43.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-50.369748
25%,0.000000,151.000000,36.000000,54.000000,29.000000,32.000000,6.444444,7.304348,-1.160884
50%,0.000000,170.000000,74.000000,94.000000,58.000000,62.000000,7.741935,9.096774,1.520000
75%,1.000000,192.000000,115.000000,134.000000,88.000000,91.000000,9.066667,11.312500,4.526786
max,1.000000,288.000000,287.000000,287.000000,121.000000,120.000000,66.000000,792.000000,785.344538


In [30]:
# Ensure balls remaining never becomes negative
df['balls_remaining'] = df['balls_remaining'].clip(lower=0)

print("Balls remaining corrected successfully!")
print("Minimum balls remaining:", df['balls_remaining'].min())

Balls remaining corrected successfully!
Minimum balls remaining: 0


In [31]:
# Batter's contribution percentage to the team's current score

df['batter_contribution_pct'] = (
    df['batter_runs'] / df['team_runs'].replace(0, np.nan)
) * 100

print("Batter contribution feature created successfully!")

df[
    [
        'batter',
        'batter_runs',
        'team_runs',
        'batter_contribution_pct'
    ]
].head(10)

Batter contribution feature created successfully!


,batter,batter_runs,team_runs,batter_contribution_pct
0,SC Ganguly,0,1,0.000000
1,BB McCullum,0,1,0.000000
2,BB McCullum,0,2,0.000000
3,BB McCullum,0,2,0.000000
4,BB McCullum,0,2,0.000000
5,BB McCullum,0,2,0.000000
6,BB McCullum,0,3,0.000000
7,BB McCullum,0,3,0.000000
8,BB McCullum,4,7,57.142857
9,BB McCullum,8,11,72.727273


In [32]:
# Feature Engineering: Batter Strike Rate

df['batter_strike_rate'] = (
    df['batter_runs'] /
    df['batter_balls'].replace(0, np.nan)
) * 100

print("Batter Strike Rate feature created successfully!")

df[
    [
        'batter',
        'batter_runs',
        'batter_balls',
        'batter_strike_rate'
    ]
].head(10)

Batter Strike Rate feature created successfully!


,batter,batter_runs,batter_balls,batter_strike_rate
0,SC Ganguly,0,1,0.000000
1,BB McCullum,0,1,0.000000
2,BB McCullum,0,1,0.000000
3,BB McCullum,0,2,0.000000
4,BB McCullum,0,3,0.000000
5,BB McCullum,0,4,0.000000
6,BB McCullum,0,5,0.000000
7,BB McCullum,0,6,0.000000
8,BB McCullum,4,7,57.142857
9,BB McCullum,8,8,100.000000


In [33]:
# Feature Engineering: Batter Boundary Runs

# Runs scored through boundaries on each ball
df['boundary_runs'] = (
    df['is_four'] * 4 +
    df['is_six'] * 6
)

print("Boundary runs feature created successfully!")

df[
    [
        'batter',
        'runs_batter',
        'is_four',
        'is_six',
        'boundary_runs'
    ]
].head(10)

Boundary runs feature created successfully!


,batter,runs_batter,is_four,is_six,boundary_runs
0,SC Ganguly,0,0,0,0
1,BB McCullum,0,0,0,0
2,BB McCullum,0,0,0,0
3,BB McCullum,0,0,0,0
4,BB McCullum,0,0,0,0
5,BB McCullum,0,0,0,0
6,BB McCullum,0,0,0,0
7,BB McCullum,0,0,0,0
8,BB McCullum,4,1,0,4
9,BB McCullum,4,1,0,4


In [34]:
# Feature Engineering: Cumulative Boundary Runs

df['cumulative_boundary_runs'] = (
    df.groupby(
        ['match_id', 'innings', 'batter']
    )['boundary_runs'].cumsum()
)

print("Cumulative boundary runs feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'batter',
        'boundary_runs',
        'cumulative_boundary_runs'
    ]
].head(10)

Cumulative boundary runs feature created successfully!


,match_id,innings,batter,boundary_runs,cumulative_boundary_runs
0,335982,1,SC Ganguly,0,0
1,335982,1,BB McCullum,0,0
2,335982,1,BB McCullum,0,0
3,335982,1,BB McCullum,0,0
4,335982,1,BB McCullum,0,0
5,335982,1,BB McCullum,0,0
6,335982,1,BB McCullum,0,0
7,335982,1,BB McCullum,0,0
8,335982,1,BB McCullum,4,4
9,335982,1,BB McCullum,4,8


In [35]:
# Feature Engineering: Boundary Contribution Percentage

df['boundary_contribution_pct'] = (
    df['cumulative_boundary_runs'] /
    df['batter_runs'].replace(0, np.nan)
) * 100

print("Boundary Contribution Percentage feature created successfully!")

df[
    [
        'batter',
        'batter_runs',
        'cumulative_boundary_runs',
        'boundary_contribution_pct'
    ]
].head(15)

Boundary Contribution Percentage feature created successfully!


,batter,batter_runs,cumulative_boundary_runs,boundary_contribution_pct
0,SC Ganguly,0,0,NaN
1,BB McCullum,0,0,NaN
2,BB McCullum,0,0,NaN
3,BB McCullum,0,0,NaN
4,BB McCullum,0,0,NaN
5,BB McCullum,0,0,NaN
6,BB McCullum,0,0,NaN
7,BB McCullum,0,0,NaN
8,BB McCullum,4,4,100.0
9,BB McCullum,8,8,100.0


In [36]:
# Feature Engineering: Cumulative Dot Balls Faced

df['cumulative_dot_balls'] = (
    df.groupby(
        ['match_id', 'innings', 'batter']
    )['is_dot_ball']
    .cumsum()
)

print("Cumulative dot balls feature created successfully!")

df[
    [
        'batter',
        'batter_balls',
        'is_dot_ball',
        'cumulative_dot_balls'
    ]
].head(15)

Cumulative dot balls feature created successfully!


,batter,batter_balls,is_dot_ball,cumulative_dot_balls
0,SC Ganguly,1,0,0
1,BB McCullum,1,1,1
2,BB McCullum,1,0,1
3,BB McCullum,2,1,2
4,BB McCullum,3,1,3
5,BB McCullum,4,1,4
6,BB McCullum,5,0,4
7,BB McCullum,6,1,5
8,BB McCullum,7,0,5
9,BB McCullum,8,0,5


In [37]:
# Feature Engineering: Batter Dot Ball Percentage

df['dot_ball_pct'] = (
    df['cumulative_dot_balls'] /
    df['batter_balls'].replace(0, np.nan)
) * 100

print("Dot Ball Percentage feature created successfully!")

df[
    [
        'batter',
        'batter_balls',
        'cumulative_dot_balls',
        'dot_ball_pct'
    ]
].head(15)

Dot Ball Percentage feature created successfully!


,batter,batter_balls,cumulative_dot_balls,dot_ball_pct
0,SC Ganguly,1,0,0.000000
1,BB McCullum,1,1,100.000000
2,BB McCullum,1,1,100.000000
3,BB McCullum,2,2,100.000000
4,BB McCullum,3,3,100.000000
5,BB McCullum,4,4,100.000000
6,BB McCullum,5,4,80.000000
7,BB McCullum,6,5,83.333333
8,BB McCullum,7,5,71.428571
9,BB McCullum,8,5,62.500000


In [38]:
# Feature Engineering: Strike Rotation

# Count cumulative singles scored by each batter
df['cumulative_singles'] = (
    df.groupby(['match_id', 'innings', 'batter'])['is_single']
      .cumsum()
)

# Calculate strike rotation percentage
df['strike_rotation_pct'] = (
    df['cumulative_singles'] /
    df['batter_balls'] * 100
)

print("Strike Rotation features created successfully!")

df[
    [
        'batter',
        'batter_balls',
        'is_single',
        'cumulative_singles',
        'strike_rotation_pct'
    ]
].head(15)

Strike Rotation features created successfully!


,batter,batter_balls,is_single,cumulative_singles,strike_rotation_pct
0,SC Ganguly,1,0,0,0.0
1,BB McCullum,1,0,0,0.0
2,BB McCullum,1,0,0,0.0
3,BB McCullum,2,0,0,0.0
4,BB McCullum,3,0,0,0.0
5,BB McCullum,4,0,0,0.0
6,BB McCullum,5,0,0,0.0
7,BB McCullum,6,0,0,0.0
8,BB McCullum,7,0,0,0.0
9,BB McCullum,8,0,0,0.0


In [39]:
# Find rows where the batter scored at least one single

single_test = df.loc[
    df['is_single'] == 1,
    [
        'match_id',
        'innings',
        'batter',
        'batter_balls',
        'is_single',
        'cumulative_singles',
        'strike_rotation_pct'
    ]
].head(15)

single_test

,match_id,innings,batter,batter_balls,is_single,cumulative_singles,strike_rotation_pct
17,335982,1,BB McCullum,13,1,1,7.692308
24,335982,1,SC Ganguly,8,1,1,12.500000
27,335982,1,SC Ganguly,10,1,2,20.000000
30,335982,1,BB McCullum,19,1,2,10.526316
32,335982,1,BB McCullum,20,1,3,15.000000
38,335982,1,BB McCullum,21,1,4,19.047619
39,335982,1,RT Ponting,5,1,1,20.000000
40,335982,1,BB McCullum,22,1,5,22.727273
42,335982,1,RT Ponting,7,1,2,28.571429
43,335982,1,BB McCullum,23,1,6,26.086957


In [40]:
# Feature Engineering: Bowler Balls Delivered

df['bowler_balls'] = (
    df.groupby(['match_id', 'innings', 'bowler'])['valid_ball']
      .cumsum()
)

print("Bowler Balls feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'bowler',
        'valid_ball',
        'bowler_balls'
    ]
].head(15)

Bowler Balls feature created successfully!


,match_id,innings,bowler,valid_ball,bowler_balls
0,335982,1,P Kumar,1,1
1,335982,1,P Kumar,1,2
2,335982,1,P Kumar,0,2
3,335982,1,P Kumar,1,3
4,335982,1,P Kumar,1,4
5,335982,1,P Kumar,1,5
6,335982,1,P Kumar,1,6
7,335982,1,Z Khan,1,1
8,335982,1,Z Khan,1,2
9,335982,1,Z Khan,1,3


In [41]:
# Feature Engineering: Bowler Runs Conceded

# Calculate cumulative runs conceded by each bowler
df['bowler_runs_conceded'] = (
    df.groupby(['match_id', 'innings', 'bowler'])['runs_bowler']
      .cumsum()
)

print("Bowler Runs Conceded feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'bowler',
        'runs_bowler',
        'bowler_runs_conceded'
    ]
].head(15)

Bowler Runs Conceded feature created successfully!


,match_id,innings,bowler,runs_bowler,bowler_runs_conceded
0,335982,1,P Kumar,0,0
1,335982,1,P Kumar,0,0
2,335982,1,P Kumar,1,1
3,335982,1,P Kumar,0,1
4,335982,1,P Kumar,0,1
5,335982,1,P Kumar,0,1
6,335982,1,P Kumar,0,1
7,335982,1,Z Khan,0,0
8,335982,1,Z Khan,4,4
9,335982,1,Z Khan,4,8


In [42]:
# Feature Engineering: Bowler Economy Rate

df['bowler_economy_rate'] = (
    df['bowler_runs_conceded'] /
    df['bowler_balls']
) * df['balls_per_over']

# Replace possible infinite values with NaN
df['bowler_economy_rate'] = (
    df['bowler_economy_rate']
    .replace([np.inf, -np.inf], np.nan)
)

print("Bowler Economy Rate feature created successfully!")

df[
    [
        'bowler',
        'bowler_balls',
        'bowler_runs_conceded',
        'balls_per_over',
        'bowler_economy_rate'
    ]
].head(15)

Bowler Economy Rate feature created successfully!


,bowler,bowler_balls,bowler_runs_conceded,balls_per_over,bowler_economy_rate
0,P Kumar,1,0,6,0.000000
1,P Kumar,2,0,6,0.000000
2,P Kumar,2,1,6,3.000000
3,P Kumar,3,1,6,2.000000
4,P Kumar,4,1,6,1.500000
5,P Kumar,5,1,6,1.200000
6,P Kumar,6,1,6,1.000000
7,Z Khan,1,0,6,0.000000
8,Z Khan,2,4,6,12.000000
9,Z Khan,3,8,6,16.000000


In [43]:
# Feature Engineering: Bowler Wickets

# Calculate cumulative wickets credited to each bowler
df['cumulative_bowler_wickets'] = (
    df.groupby(['match_id', 'innings', 'bowler'])['bowler_wicket']
      .cumsum()
)

print("Cumulative Bowler Wickets feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'bowler',
        'bowler_wicket',
        'cumulative_bowler_wickets'
    ]
].head(20)

Cumulative Bowler Wickets feature created successfully!


,match_id,innings,bowler,bowler_wicket,cumulative_bowler_wickets
0,335982,1,P Kumar,0,0
1,335982,1,P Kumar,0,0
2,335982,1,P Kumar,0,0
3,335982,1,P Kumar,0,0
4,335982,1,P Kumar,0,0
5,335982,1,P Kumar,0,0
6,335982,1,P Kumar,0,0
7,335982,1,Z Khan,0,0
8,335982,1,Z Khan,0,0
9,335982,1,Z Khan,0,0


In [44]:
# Check rows where the bowler has taken a wicket

df.loc[
    df['bowler_wicket'] == 1,
    [
        'match_id',
        'innings',
        'bowler',
        'bowler_wicket',
        'cumulative_bowler_wickets'
    ]
].head(20)

,match_id,innings,bowler,bowler_wicket,cumulative_bowler_wickets
33,335982,1,Z Khan,1,1
74,335982,1,JH Kallis,1,1
106,335982,1,AA Noffke,1,1
131,335982,2,I Sharma,1,1
138,335982,2,AB Dinda,1,1
154,335982,2,AB Agarkar,1,1
157,335982,2,AB Dinda,1,2
174,335982,2,SC Ganguly,1,1
177,335982,2,AB Agarkar,1,2
183,335982,2,AB Agarkar,1,3


In [45]:
# Feature Engineering: Bowler Dot Balls

df['cumulative_bowler_dot_balls'] = (
    df.groupby(['match_id', 'innings', 'bowler'])['is_dot_ball']
      .cumsum()
)

print("Cumulative Bowler Dot Balls feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'bowler',
        'bowler_balls',
        'is_dot_ball',
        'cumulative_bowler_dot_balls'
    ]
].head(15)

Cumulative Bowler Dot Balls feature created successfully!


,match_id,innings,bowler,bowler_balls,is_dot_ball,cumulative_bowler_dot_balls
0,335982,1,P Kumar,1,0,0
1,335982,1,P Kumar,2,1,1
2,335982,1,P Kumar,2,0,1
3,335982,1,P Kumar,3,1,2
4,335982,1,P Kumar,4,1,3
5,335982,1,P Kumar,5,1,4
6,335982,1,P Kumar,6,0,4
7,335982,1,Z Khan,1,1,1
8,335982,1,Z Khan,2,0,1
9,335982,1,Z Khan,3,0,1


In [46]:
# Feature Engineering: Bowler Dot Ball Percentage

df['bowler_dot_ball_pct'] = (
    df['cumulative_bowler_dot_balls']
    / df['bowler_balls']
) * 100

print("Bowler Dot Ball Percentage feature created successfully!")

df[
    [
        'bowler',
        'bowler_balls',
        'cumulative_bowler_dot_balls',
        'bowler_dot_ball_pct'
    ]
].head(15)

Bowler Dot Ball Percentage feature created successfully!


,bowler,bowler_balls,cumulative_bowler_dot_balls,bowler_dot_ball_pct
0,P Kumar,1,0,0.000000
1,P Kumar,2,1,50.000000
2,P Kumar,2,1,50.000000
3,P Kumar,3,2,66.666667
4,P Kumar,4,3,75.000000
5,P Kumar,5,4,80.000000
6,P Kumar,6,4,66.666667
7,Z Khan,1,1,100.000000
8,Z Khan,2,1,50.000000
9,Z Khan,3,1,33.333333


In [47]:
# Feature Engineering: Bowler Boundary Runs Conceded

df['bowler_boundary_runs'] = (
    df['runs_batter']
    * df['is_boundary']
)

print("Bowler Boundary Runs feature created successfully!")

df[
    [
        'bowler',
        'runs_batter',
        'is_four',
        'is_six',
        'is_boundary',
        'bowler_boundary_runs'
    ]
].head(15)

Bowler Boundary Runs feature created successfully!


,bowler,runs_batter,is_four,is_six,is_boundary,bowler_boundary_runs
0,P Kumar,0,0,0,0,0
1,P Kumar,0,0,0,0,0
2,P Kumar,0,0,0,0,0
3,P Kumar,0,0,0,0,0
4,P Kumar,0,0,0,0,0
5,P Kumar,0,0,0,0,0
6,P Kumar,0,0,0,0,0
7,Z Khan,0,0,0,0,0
8,Z Khan,4,1,0,1,4
9,Z Khan,4,1,0,1,4


In [48]:
# Feature Engineering: Cumulative Bowler Boundary Runs

df['cumulative_bowler_boundary_runs'] = (
    df.groupby(['match_id', 'innings', 'bowler'])['bowler_boundary_runs']
      .cumsum()
)

print("Cumulative Bowler Boundary Runs feature created successfully!")

df[
    [
        'match_id',
        'innings',
        'bowler',
        'bowler_boundary_runs',
        'cumulative_bowler_boundary_runs'
    ]
].head(20)

Cumulative Bowler Boundary Runs feature created successfully!


,match_id,innings,bowler,bowler_boundary_runs,cumulative_bowler_boundary_runs
0,335982,1,P Kumar,0,0
1,335982,1,P Kumar,0,0
2,335982,1,P Kumar,0,0
3,335982,1,P Kumar,0,0
4,335982,1,P Kumar,0,0
5,335982,1,P Kumar,0,0
6,335982,1,P Kumar,0,0
7,335982,1,Z Khan,0,0
8,335982,1,Z Khan,4,4
9,335982,1,Z Khan,4,8


In [49]:
# Feature Engineering: Bowler Boundary Concession Percentage

df['bowler_boundary_concession_pct'] = np.where(
    df['bowler_runs_conceded'] > 0,
    
    (
        df['cumulative_bowler_boundary_runs']
        / df['bowler_runs_conceded']
    ) * 100,
    
    0
)

print("Bowler Boundary Concession Percentage feature created successfully!")

df[
    [
        'bowler',
        'bowler_runs_conceded',
        'cumulative_bowler_boundary_runs',
        'bowler_boundary_concession_pct'
    ]
].head(20)

Bowler Boundary Concession Percentage feature created successfully!


,bowler,bowler_runs_conceded,cumulative_bowler_boundary_runs,bowler_boundary_concession_pct
0,P Kumar,0,0,0.000000
1,P Kumar,0,0,0.000000
2,P Kumar,1,0,0.000000
3,P Kumar,1,0,0.000000
4,P Kumar,1,0,0.000000
5,P Kumar,1,0,0.000000
6,P Kumar,1,0,0.000000
7,Z Khan,0,0,0.000000
8,Z Khan,4,4,100.000000
9,Z Khan,8,8,100.000000


In [50]:
# Create match phase feature based on T20 overs

def get_match_phase(over):
    if over < 6:
        return "Powerplay"
    elif over < 15:
        return "Middle Overs"
    else:
        return "Death Overs"


df['match_phase'] = df['over'].apply(get_match_phase)

print("Match Phase feature created successfully!")

Match Phase feature created successfully!


In [51]:
print(df['match_phase'].value_counts())

df[
    [
        'match_id',
        'innings',
        'over',
        'ball',
        'match_phase'
    ]
].head(30)

match_phase
Middle Overs    135655
Powerplay        92904
Death Overs      67173
Name: count, dtype: int64


,match_id,innings,over,ball,match_phase
0,335982,1,0,1,Powerplay
1,335982,1,0,2,Powerplay
2,335982,1,0,3,Powerplay
3,335982,1,0,3,Powerplay
4,335982,1,0,4,Powerplay
5,335982,1,0,5,Powerplay
6,335982,1,0,6,Powerplay
7,335982,1,1,1,Powerplay
8,335982,1,1,2,Powerplay
9,335982,1,1,3,Powerplay


In [52]:
# Create binary match phase indicators

df['is_powerplay'] = (
    df['match_phase'] == 'Powerplay'
).astype(int)

df['is_middle_overs'] = (
    df['match_phase'] == 'Middle Overs'
).astype(int)

df['is_death_overs'] = (
    df['match_phase'] == 'Death Overs'
).astype(int)

print("Binary Match Phase features created successfully!")

Binary Match Phase features created successfully!


In [53]:
df[
    [
        'over',
        'match_phase',
        'is_powerplay',
        'is_middle_overs',
        'is_death_overs'
    ]
].drop_duplicates().sort_values('over')

,over,match_phase,is_powerplay,is_middle_overs,is_death_overs
0,0,Powerplay,1,0,0
7,1,Powerplay,1,0,0
13,2,Powerplay,1,0,0
19,3,Powerplay,1,0,0
26,4,Powerplay,1,0,0
32,5,Powerplay,1,0,0
38,6,Middle Overs,0,1,0
44,7,Middle Overs,0,1,0
50,8,Middle Overs,0,1,0
56,9,Middle Overs,0,1,0


In [54]:
# Cumulative batter runs within each match phase

df['batter_phase_runs'] = (
    df.groupby(
        [
            'match_id',
            'innings',
            'batter',
            'match_phase'
        ]
    )['runs_batter']
    .cumsum()
)

print("Batter Phase Runs feature created successfully!")

Batter Phase Runs feature created successfully!


In [55]:
df[
    [
        'match_id',
        'innings',
        'match_phase',
        'batter',
        'runs_batter',
        'batter_phase_runs'
    ]
].head(25)

,match_id,innings,match_phase,batter,runs_batter,batter_phase_runs
0,335982,1,Powerplay,SC Ganguly,0,0
1,335982,1,Powerplay,BB McCullum,0,0
2,335982,1,Powerplay,BB McCullum,0,0
3,335982,1,Powerplay,BB McCullum,0,0
4,335982,1,Powerplay,BB McCullum,0,0
5,335982,1,Powerplay,BB McCullum,0,0
6,335982,1,Powerplay,BB McCullum,0,0
7,335982,1,Powerplay,BB McCullum,0,0
8,335982,1,Powerplay,BB McCullum,4,4
9,335982,1,Powerplay,BB McCullum,4,8


In [56]:
# Cumulative valid balls faced by batter within each match phase

df['batter_phase_balls'] = (
    df.groupby(
        [
            'match_id',
            'innings',
            'batter',
            'match_phase'
        ]
    )['valid_ball']
    .cumsum()
)

print("Batter Phase Balls feature created successfully!")

Batter Phase Balls feature created successfully!


In [57]:
df[
    [
        'match_id',
        'innings',
        'match_phase',
        'batter',
        'valid_ball',
        'batter_phase_balls'
    ]
].head(25)

,match_id,innings,match_phase,batter,valid_ball,batter_phase_balls
0,335982,1,Powerplay,SC Ganguly,1,1
1,335982,1,Powerplay,BB McCullum,1,1
2,335982,1,Powerplay,BB McCullum,0,1
3,335982,1,Powerplay,BB McCullum,1,2
4,335982,1,Powerplay,BB McCullum,1,3
5,335982,1,Powerplay,BB McCullum,1,4
6,335982,1,Powerplay,BB McCullum,1,5
7,335982,1,Powerplay,BB McCullum,1,6
8,335982,1,Powerplay,BB McCullum,1,7
9,335982,1,Powerplay,BB McCullum,1,8


In [58]:
import numpy as np

# Calculate batter strike rate within each match phase

df['batter_phase_strike_rate'] = np.where(
    df['batter_phase_balls'] > 0,
    (df['batter_phase_runs'] / df['batter_phase_balls']) * 100,
    np.nan
)

print("Batter Phase Strike Rate feature created successfully!")

# Check infinite values
print(
    "Infinite values:",
    np.isinf(df['batter_phase_strike_rate']).sum()
)

Batter Phase Strike Rate feature created successfully!
Infinite values: 0


In [59]:
df[
    [
        'match_id',
        'innings',
        'match_phase',
        'batter',
        'batter_phase_runs',
        'batter_phase_balls',
        'batter_phase_strike_rate'
    ]
].head(25)

,match_id,innings,match_phase,batter,batter_phase_runs,batter_phase_balls,batter_phase_strike_rate
0,335982,1,Powerplay,SC Ganguly,0,1,0.000000
1,335982,1,Powerplay,BB McCullum,0,1,0.000000
2,335982,1,Powerplay,BB McCullum,0,1,0.000000
3,335982,1,Powerplay,BB McCullum,0,2,0.000000
4,335982,1,Powerplay,BB McCullum,0,3,0.000000
5,335982,1,Powerplay,BB McCullum,0,4,0.000000
6,335982,1,Powerplay,BB McCullum,0,5,0.000000
7,335982,1,Powerplay,BB McCullum,0,6,0.000000
8,335982,1,Powerplay,BB McCullum,4,7,57.142857
9,335982,1,Powerplay,BB McCullum,8,8,100.000000
